# 🧠 MURE Custom LLM Training — ONE-CLICK Colab Notebook
> **သင်ယူမှုအသစ်: ကိုယ်ပိုင် Sentence-based LLM (MURE SL-3B) ကို 5,000,000 Priming Data ဖြင့် သုညမှစတင်၍ Train ပါမည်။**

### ⚡ အဆင့်ဆင့်လုပ်ဆောင်ချက်:
1. Runtime → Change runtime type → **T4 GPU** ✅
2. Runtime → **Run All** ✅

### 📊 System Trace & Architecture:
- **Tokenizer**: Gemma-2B Fast BPE (256,000 vocab)
- **Model**: Causal Transformer Graph (1024-dim, 12-layer, 16-head)
- **Dataset**: 5,000,000 causal logic entries (JSONL format)
- **Training**: Next-token prediction (Autoregressive)
- **Persistence**: Automatic State Saving to Google Drive

# 🚀 PHASE 1 — ENVIRONMENT & PERSISTENCE SETUP

In [ ]:
from google.colab import drive
import os, torch, json, random, time
from tqdm.auto import tqdm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader

# 1. Google Drive Mount (Data & Model သိမ်းရန်)
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)

DRIVE_FOLDER = '/content/drive/MyDrive/mure_llm_workspace'
os.makedirs(DRIVE_FOLDER, exist_ok=True)

# 2. Dependencies Install
!pip install transformers accelerate sentencepiece datasets
from transformers import AutoTokenizer

print("\n✅ Environment Ready.")
if torch.cuda.is_available():
    print(f"✅ GPU Enabled: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("⛔ GPU မတွေ့ပါ။ Runtime → T4 GPU ကို ရွေးချယ်ပါ။")

# 📦 PHASE 2 — GENERATE 5,000,000 CAUSAL REASONING RECORDS

In [ ]:
OUTPUT_JSONL = os.path.join(DRIVE_FOLDER, 'mure_data_5M.jsonl')

CAUSAL_RULES = [
    {"c": "high humidity", "e": "formation of clouds", "s": 0.92},
    {"c": "burning fossil fuels", "e": "release of CO2", "s": 0.98},
    {"c": "volcano eruption", "e": "ash clouds blocking sunlight", "s": 0.95},
    {"c": "မိုးသည်းထန်စွာရွာသွန်းခြင်း", "e": "မြေနိမ့်ပိုင်းများ ရေကြီးခြင်း", "s": 0.88},
    {"c": "ပုံမှန်လေ့ကျင့်ခန်းလုပ်ခြင်း", "e": "နှလုံးရောဂါဖြစ်နိုင်ခြေလျော့ကျခြင်း", "s": 0.94},
    {"c": "increase in demand", "e": "rise in market price", "s": 0.85},
    {"c": "solar wind hitting magnetosphere", "e": "aurora borealis phenomenon", "s": 0.99}
]

TEMPLATES = [
    "Question: What is the result of {c}? Answer: The result is {e}.",
    "If {c} happens, then {e} will occur.",
    "Trace effect: {c} leads to {e}.",
    "{c} ဆိုတာ ဘာကို ဖြစ်စေသလဲ? အဖြေ - {e} ကို ဖြစ်စေပါသည်။"
]

def generate_dataset_trace(target=5000000):
    if os.path.exists(OUTPUT_JSONL):
        print(f"✅ Dataset already exists at {OUTPUT_JSONL}")
        return
        
    print(f"🚀 Generating 5,000,000 priming records...")
    with open(OUTPUT_JSONL, 'w', encoding='utf-8') as f:
        for i in range(target):
            rule = random.choice(CAUSAL_RULES)
            fmt = random.choice(TEMPLATES)
            text = fmt.format(c=rule['c'], e=rule['e'])
            json_line = json.dumps({"text": text}, ensure_ascii=False)
            f.write(json_line + '\n')
            if (i+1) % 1000000 == 0: 
                print(f"   ... {i+1:,} entries generated")
    print("✅ Dataset Generation Complete.")

generate_dataset_trace()

# 🧠 PHASE 3 — MURE SL-3B CUSTOM ARCHITECTURE
> **This is not a fine-tune. We are training a scratch model using a Causal Transformer.**

In [ ]:
TOKENIZER_ID = "google/gemma-2b" # Using Gemma's tokenizer for optimized speed/vocab
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
tokenizer.pad_token = tokenizer.eos_token

class MURE_SentenceLLM(nn.Module):
    def __init__(self, vocab_size, d_model=1024, n_layers=12, n_heads=16):
        super(MURE_SentenceLLM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = nn.Parameter(torch.zeros(1, 1024, d_model))
        
        # Encoder Layer with causal capability
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=n_heads, 
            dim_feedforward=4096, 
            batch_first=True, 
            norm_first=True,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x, mask=None):
        sz = x.size(1)
        if mask is None:
            # Causal Triangular Mask (Crucial for Autoregressive Training)
            mask = torch.triu(torch.ones(sz, sz) * float('-inf'), diagonal=1).to(x.device)
        
        h = self.embedding(x) + self.pos_encoding[:, :sz, :]
        h = self.transformer(h, mask=mask)
        return self.lm_head(h)

model = MURE_SentenceLLM(vocab_size=tokenizer.vocab_size).cuda()
params = sum(p.numel() for p in model.parameters())
print(f"✅ MURE SL-3B Ready. Total Parameters: {params:,}")

# 🏋️ PHASE 4 — EPOCH 1: 5-MILLION ENTRY PRIMING

In [ ]:
class MUREDataset(IterableDataset):
    def __init__(self, file_path, tokenizer, max_len=64):
        self.file_path = file_path
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __iter__(self):
        with open(self.file_path, 'r', encoding='utf-8') as f:
            for line in f:
                text = json.loads(line)['text']
                tokens = self.tokenizer(text, truncation=True, max_length=self.max_len, padding='max_length', return_tensors="pt")
                yield tokens['input_ids'].squeeze(0)

dataset = MUREDataset(OUTPUT_JSONL, tokenizer)
dataloader = DataLoader(dataset, batch_size=40) # T4 optimized batch size

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
checkpoint_path = os.path.join(DRIVE_FOLDER, 'mure_sl3b_checkpoint.pt')

print("🚀 Starting Scratch Training (Epoch 1)...")
model.train()
step = 0
pbar = tqdm(dataloader, total=5000000//40, desc="Training MURE SL-3B")

for batch in pbar:
    batch = batch.cuda()
    
    # Shift inputs/targets for causal modeling
    inputs = batch[:, :-1]
    targets = batch[:, 1:]
    
    optimizer.zero_grad()
    logits = model(inputs)
    
    # Flatten for loss calculation
    loss = criterion(logits.reshape(-1, tokenizer.vocab_size), targets.reshape(-1))
    loss.backward()
    
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    step += 1
    if step % 50 == 0:
        pbar.set_postfix({"Loss": f"{loss.item():.4f}"})
    
    if step % 10000 == 0:
        torch.save(model.state_dict(), checkpoint_path)
        print(f"\n💾 Auto-save at step {step}")

torch.save(model.state_dict(), os.path.join(DRIVE_FOLDER, 'mure_sl3b_final.pt'))
print("\n✨ TRAINING COMPLETE. Model saved to Google Drive.")